# SWED: supervisor tasks on the real data

Run this notebook **on the cluster**, next to the real Parquet files. It covers the six tasks:

| § | Task |
|---|---|
| 1 | Throw out Tolvan Tolvansson (test patient) |
| 2 | Inventory of vital signs in every EHR file |
| 3 | Map vital signs to SNOMED |
| 4 | Are vital signs measured in the same units? How does Sahlgrenska do it? |
| 5 | Death register: variable list against our file |
| 6 | Load diagnoses from the patient register (fat node) |
| 7 | Validate the synthetic dataset against the real data |

## Rules for sensitive data

* Everything here stays on the cluster. Only aggregates leave, and only after the project's export check.
* `show()` hides groups with fewer than `MIN_CELL` persons or rows. Use it for anything you might export.
* Notebook outputs are saved **inside the .ipynb file**. Before you move this notebook off the cluster, clear them:
  `jupyter nbconvert --clear-output --inplace swed_tasks.ipynb`
* Don't paste row-level output into chat tools (including Claude). Aggregates that passed `show()` are fine.

Files to copy to the cluster with this notebook: `data_info/*.csv` and `make_synthetic_swed.py`.

## 0. Setup

In [ ]:
import os, re, resource
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

DATA = Path(os.environ.get("SWED_DATA", "/proj/nobackup/sens2020598/swed_db/clean_parquet"))
WORK = Path(os.environ.get("SWED_WORK", Path.cwd()))   # folder with this notebook
REF = WORK / "data_info"                                # the mapping CSVs
OUT = WORK / "out"                                      # results (still sensitive until checked)
OUT.mkdir(exist_ok=True)
MIN_CELL = 10                                           # smallest group shown

pd.set_option("display.max_rows", 200, "display.max_columns", 50, "display.width", 200)
print("pyarrow", pa.__version__, "| pandas", pd.__version__)
print("files:", sorted(p.stem for p in DATA.glob("*.parquet")))

In [ ]:
def path(name):
    return DATA / f"{name}.parquet"

def have(name):
    ok = path(name).exists()
    if not ok:
        print(f"skip: {name}.parquet not found")
    return ok

def info(name):
    m = pq.read_metadata(path(name))
    print(f"{name}: {m.num_rows:,} rows, {m.num_row_groups} row groups")
    print(pq.read_schema(path(name)).to_string(show_schema_metadata=False))

def scan(name, columns, filter=None, batch_size=2_000_000):
    # Yields pandas batches, so 80M-row files never sit in memory at once.
    sc = ds.dataset(path(name), format="parquet").scanner(columns=columns, filter=filter, batch_size=batch_size)
    for b in sc.to_batches():
        if b.num_rows:
            yield b.to_pandas()

def count_by(name, cols, filter=None):
    # Row counts per combination of cols, streamed.
    parts = [df.groupby(cols, dropna=False).size() for df in scan(name, cols, filter)]
    if not parts:
        return pd.DataFrame(columns=cols + ["n"])
    s = pd.concat(parts).groupby(level=list(range(len(cols))), dropna=False).sum()
    return s.rename("n").reset_index().sort_values("n", ascending=False, ignore_index=True)

def show(df, n="n", k=None, keys=()):
    # Small-cell suppression: rows with n < k get every number blanked (except grouping columns in keys).
    k = MIN_CELL if k is None else k
    out = df.copy()
    small = out[n] < k
    num = [c for c in out.select_dtypes("number").columns if c not in keys]
    out[num] = out[num].astype("float").mask(small)
    if small.any():
        print(f"({small.sum()} groups with n < {k} hidden)")
    return out

def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6   # peak RSS, Linux reports kB

## 1. Throw out Tolvan Tolvansson

Tolvan Tolvansson is the Swedish test patient (personnummer 19121212-1212). EHR systems keep test patients like him. The data is pseudonymised, so we can't search for the personnummer. Instead we look for his traces:

1. **Birth date 1912-12-12** in `swed_ed_core_birthdate`.
2. **No SCB record**: a test patient isn't a real resident, so he shouldn't be in `scb_background`.
3. **Extreme row counts**: test patients often get hundreds of test visits or lab rows.

Ask your supervisor whether they already know his `lopnr_swed`. If they do, put it straight into `EXCLUDE` below.

In [ ]:
cand = pd.DataFrame(columns=["lopnr_swed"])
if have("swed_ed_core_birthdate"):
    bd = pq.read_table(path("swed_ed_core_birthdate")).to_pandas()
    cand = bd.loc[bd["birth_date"] == date(1912, 12, 12), ["lopnr_swed"]].drop_duplicates()
    print(f"born 1912-12-12: {len(cand)} person(s)")

if have("scb_background") and len(cand):
    scb_ids = set(pq.read_table(path("scb_background"), columns=["lopnr_swed"])["lopnr_swed"].to_pylist())
    cand["in_scb"] = cand["lopnr_swed"].isin(scb_ids)

if have("swed_ed_core") and len(cand):
    ed = pq.read_table(path("swed_ed_core"), columns=["lopnr_swed", "ed_hospital", "ed_arrival_date", "ed_age"],
                       filters=[("lopnr_swed", "in", cand["lopnr_swed"].tolist())]).to_pandas()
    per = ed.groupby("lopnr_swed").agg(ed_visits=("ed_hospital", "size"),
                                       hospitals=("ed_hospital", lambda s: ", ".join(sorted(s.unique()))),
                                       first=("ed_arrival_date", "min"), last=("ed_arrival_date", "max"),
                                       age_min=("ed_age", "min"), age_max=("ed_age", "max"))
    cand = cand.merge(per, on="lopnr_swed", how="left")
cand

In [ ]:
# Who has the most rows? A test patient often stands out by far.
# Persons are only kept at the top of the list, so nothing small is shown.
TOP = 10
for name in ["swed_ed_core", "ehr_danderyd_lab_meas", "ehr_karolinska_meas", "ehr_sahlgrenska_lab", "ehr_sa_vgr_lab"]:
    if not have(name):
        continue
    c = count_by(name, ["lopnr_swed"])
    c["born_1912_12_12"] = c["lopnr_swed"].isin(cand["lopnr_swed"])
    print(f"\n{name}: {len(c):,} persons, median {c['n'].median():.0f} rows/person, "
          f"rows without lopnr: {c.loc[c['lopnr_swed'].isna(), 'n'].sum():,}")
    display(c.head(TOP))

In [ ]:
# Decide by hand, then save. Keep the reason so you can explain it in the methods section.
EXCLUDE = {
    # 1234567.0: "born 1912-12-12, not in SCB, 800 ED visits",
}
for lop in cand["lopnr_swed"]:
    print("candidate:", lop, "(add to EXCLUDE if you agree)")

pd.DataFrame({"lopnr_swed": list(EXCLUDE), "reason": list(EXCLUDE.values())}).to_csv(OUT / "exclude_lopnr.csv", index=False)

def not_excluded():
    # Pass as filter= to scan()/count_by()/pq.read_table() in every later analysis.
    return ~ds.field("lopnr_swed").isin(list(EXCLUDE)) if EXCLUDE else None

## 2. Vital-sign inventory

Vital signs live in different files depending on the hospital:

| File | Unit column | Note from the profile |
|---|---|---|
| `ehr_danderyd_lab_meas` | yes | Danderyd + SÖS. Vital signs have **no unit** (null). |
| `ehr_karolinska_meas` | yes | Vital signs with units: mmHg, %, slag/min, °C, andetag/min. |
| `ehr_sa_vgr_lab` | yes | Blodtryck, Puls, Temperatur, Saturation. 69% of all units are null. |
| `ehr_sahlgrenska_lab` | **no** | Has `term_id`, `method`, `ref_val1` instead. |
| `ehr_sos_lab`, `ehr_stgoran_lab`, `ehr_karolinska_lab` | yes | Mostly lab tests. Checked anyway. |

Step 1 counts every `term_name` per file. Step 2 picks the vital-sign candidates with a regex. **Read through the candidate list by hand**: the regex has false hits, e.g. a blood gas "corrected to patient temperature".

In [ ]:
SOURCES = ["ehr_danderyd_lab_meas", "ehr_karolinska_meas", "ehr_sa_vgr_lab", "ehr_sahlgrenska_lab",
           "ehr_sos_lab", "ehr_stgoran_lab", "ehr_karolinska_lab"]

terms = []
for name in SOURCES:
    if not have(name):
        continue
    cols = pq.read_schema(path(name)).names
    by = ["term_name"] + (["unit"] if "unit" in cols else [])
    c = count_by(name, by, not_excluded())
    c.insert(0, "source", name)
    terms.append(c)
    print(f"{name}: {c['term_name'].nunique()} terms, {c['n'].sum():,} rows")
terms = pd.concat(terms, ignore_index=True)
terms.to_csv(OUT / "all_terms_by_source.csv", index=False)

In [ ]:
VITAL_RE = re.compile(r"blodtryck|systol|diastol|\bpuls|hjärtfrekv|saturation|spo2|andning|temp|medvetande|retts|rls"
                      r"|gcs|glasgow|news|smärt|\bvas\b|\bnrs\b|vikt|längd|bmi|syrgas", re.I)
LAB_PREFIX = re.compile(r"^(?:a|v|c|k)?B-|^(?:f?P|S|U|Pt)-|^P/S-|^Csv-|^Ery-", re.I)   # blood, plasma, serum, urine ...

names = terms.groupby("term_name")["n"].sum().reset_index()
names["vital_regex"] = names["term_name"].str.contains(VITAL_RE)
names["lab_prefix"] = names["term_name"].str.contains(LAB_PREFIX)
cands = names[names["vital_regex"]].sort_values("n", ascending=False)
print("candidates with a lab prefix (probably NOT vitals):")
display(cands[cands["lab_prefix"]])
print("candidates:")
display(cands[~cands["lab_prefix"]])

In [ ]:
# Edit these two lists after reading the candidates above.
VITAL_DROP = []          # regex hits that aren't vital signs
VITAL_ADD = []           # vital signs the regex missed
VITAL_TERMS = sorted((set(cands.loc[~cands["lab_prefix"], "term_name"]) - set(VITAL_DROP)) | set(VITAL_ADD))
len(VITAL_TERMS), VITAL_TERMS

In [ ]:
# Value statistics per (source, term, unit). Only vital rows are loaded, so this fits on a normal node.
QS = [0.01, 0.05, 0.5, 0.95, 0.99]
stats = []
for name in terms["source"].unique():
    cols = pq.read_schema(path(name)).names
    extra = [c for c in ["unit", "term_id", "method"] if c in cols]
    filt = ds.field("term_name").isin(VITAL_TERMS)
    if EXCLUDE:
        filt = filt & not_excluded()
    parts = list(scan(name, ["lopnr_swed", "term_name", "value_num"] + extra, filt))
    if not parts:
        continue
    df = pd.concat(parts, ignore_index=True)
    if "unit" not in df:
        df["unit"] = pd.NA
    g = df.groupby(["term_name", "unit"], dropna=False)
    s = g["value_num"].quantile(QS).unstack()
    s.columns = [f"q{int(q * 100):02d}" for q in QS]
    s.insert(0, "n", g.size())
    s.insert(1, "persons", g["lopnr_swed"].nunique())
    s.insert(2, "value_null_%", g["value_num"].apply(lambda v: round(100 * v.isna().mean(), 2)))
    s = s.reset_index()
    s.insert(0, "source", name)
    stats.append(s)
    print(f"{name}: {len(df):,} vital rows, peak memory {mem_gb():.1f} GB")
    del df, parts
stats = pd.concat(stats, ignore_index=True)
vital_stats = show(stats, n="persons")
vital_stats.to_csv(OUT / "vitals_stats_by_source.csv", index=False)
vital_stats

## 3. Map vital signs to SNOMED

`swed_lab_meas_to_snomed.csv` already has rows for several vital signs (Blodtryck, Puls, Saturation, Temperatur, Andningsfrekvens, Medvetandegrad ...). The first step is to see which of our terms it already covers.

**Something to raise with your supervisor:** the existing file mixes two kinds of SNOMED concepts.

* For blood pressure it uses the **observable entity** (271649006 *Systolic blood pressure*). That concept means "the value".
* For pulse, temperature and saturation it uses **procedures** (65653002 *pulsmätning*, 56342008 *temperaturmätning*, 284032008 *mätning av syresättning*). Those mean "the act of measuring".

A table of measured values normally uses observable entities. The column `snomed_observable` below suggests one per vital sign.

**Verify every code in the SNOMED browser (Swedish edition) before using it.** I wrote these from memory, not from a lookup.

| Vital sign | Suggested observable | Concept |
|---|---|---|
| systolic BP | 271649006 | Systolic blood pressure |
| diastolic BP | 271650006 | Diastolic blood pressure |
| pulse | 78564009 | Pulse rate |
| heart rate (monitor/ECG) | 364075005 | Heart rate |
| respiratory rate | 86290005 | Respiratory rate |
| temperature | 386725007 | Body temperature |
| saturation (SpO2) | 431314004 | Peripheral oxygen saturation |
| level of consciousness (RLS-85) | 6942003 | Level of consciousness. RLS-85 itself may lack an international concept. |
| GCS total | 248241002 | Glasgow coma score |
| pain score (NRS/VAS) | 225908003 | Pain score |
| weight | 27113001 | Body weight |
| height | 50373000 | Body height |
| NEWS2 total | – | Only in the UK extension as far as I know. Look it up. |

In [ ]:
SUGGEST = [  # (regex on term_name, concept key, observable code)
    (r"systol",                     "systolic_bp",     "271649006"),
    (r"diastol",                    "diastolic_bp",    "271650006"),
    (r"hjärtfrekv",                 "heart_rate",      "364075005"),
    (r"\bpuls",                     "pulse_rate",      "78564009"),
    (r"andning",                    "resp_rate",       "86290005"),
    (r"temp",                       "body_temp",       "386725007"),
    (r"saturation|spo2",            "spo2",            "431314004"),
    (r"medvetande|retts|rls",       "consciousness",   "6942003"),
    (r"glasgow|gcs.*(tot|summa)|^gcs$", "gcs_total",       "248241002"),
    (r"smärt|\bvas\b|\bnrs\b",      "pain_score",      "225908003"),
    (r"vikt",                       "body_weight",     "27113001"),
    (r"längd",                      "body_height",     "50373000"),
    (r"news",                       "news_total",      ""),
    (r"syrgas",                     "oxygen",          ""),
]

def suggest(term):
    for rx, key, code_ in SUGGEST:
        if re.search(rx, term, re.I):
            return key, code_
    return "", ""

lab_map = pd.read_csv(REF / "swed_lab_meas_to_snomed.csv", dtype=str)
per_term = (stats.groupby("term_name")
                 .agg(n=("n", "sum"), sources=("source", lambda s: ", ".join(sorted(set(s)))))
                 .reset_index())
vmap = per_term.merge(lab_map.rename(columns={"snomed_id": "snomed_existing", "snomed_swe": "snomed_existing_swe"}),
                      on="term_name", how="left")
vmap[["concept", "snomed_observable"]] = vmap["term_name"].apply(lambda t: pd.Series(suggest(t)))
vmap["needs_review"] = vmap["snomed_existing"].isna() | (vmap["snomed_observable"] == "") | vmap["problematic"].notna()
vmap = vmap.sort_values(["concept", "n"], ascending=[True, False])
vmap.to_csv(OUT / "vitals_snomed_map_draft.csv", index=False)
print(f"{vmap['snomed_existing'].notna().sum()} of {len(vmap)} terms already in swed_lab_meas_to_snomed.csv")
show(vmap)

## 4. Units per hospital, and Sahlgrenska

We put the hospitals side by side for each concept. A unit can be missing or written differently, but the **value range** still gives the unit away:

* temperature median ≈ 37 → °C (≈ 98 would be °F)
* saturation median ≈ 97 → % (≈ 0.97 would be a fraction)
* weight median ≈ 75 → kg (≈ 75 000 would be grams)

`flag` marks combinations that look wrong.

In [ ]:
EXPECT = {  # concept: (expected unit, plausible range of the median)
    "systolic_bp": ("mmHg", (90, 180)), "diastolic_bp": ("mmHg", (50, 110)),
    "pulse_rate": ("/min", (50, 120)), "heart_rate": ("/min", (50, 120)), "resp_rate": ("/min", (10, 30)),
    "body_temp": ("°C", (35, 39)), "spo2": ("%", (85, 100)), "body_weight": ("kg", (40, 120)),
    "body_height": ("cm", (140, 200)), "consciousness": ("RLS 1-8", (1, 8)), "pain_score": ("0-10", (0, 10)),
    "news_total": ("score", (0, 20)),
}
u = stats.merge(vmap[["term_name", "concept"]], on="term_name", how="left")
u["expected_unit"] = u["concept"].map(lambda c: EXPECT.get(c, ("", None))[0])

def flag(r):
    lo_hi = EXPECT.get(r["concept"], ("", None))[1]
    out = []
    if pd.isna(r["unit"]):
        out.append("no unit")
    if lo_hi and pd.notna(r["q50"]) and not (lo_hi[0] <= r["q50"] <= lo_hi[1]):
        out.append(f"median {r['q50']:g} outside {lo_hi}")
    if pd.notna(r["q99"]) and pd.notna(r["q50"]) and r["q99"] > 10 * max(r["q50"], 1):
        out.append("extreme tail")
    return "; ".join(out)

u["flag"] = u.apply(flag, axis=1)
units = show(u[["concept", "source", "term_name", "unit", "expected_unit", "n", "persons", "q01", "q50", "q99", "flag"]]
               .sort_values(["concept", "source"]), n="persons")
units.to_csv(OUT / "vitals_units_by_source.csv", index=False)
units

In [ ]:
# One row per concept: which units does each hospital use?
pivot = (u.assign(unit=u["unit"].fillna("∅"))
          .groupby(["concept", "source"])["unit"].agg(lambda s: " | ".join(sorted(set(s))))
          .unstack())
pivot

### Sahlgrenska

`ehr_sahlgrenska_lab` has **no unit column**. Questions to answer:

1. Does the file contain vital signs at all, or only lab tests? If none of the candidates in §2 come from it, the vital signs for Gothenburg are probably in `ehr_sa_vgr_lab`. Find out what "sa_vgr" stands for: Sahlgrenska/VGR, or Södra Älvsborg?
2. Where does the unit information live instead: in `term_id` (e.g. `AB PO2 P2`), in `method`, or only implicitly via the reference range `ref_val1`?
3. `star_flag` is true on 27% of rows and `inexact_time` on 15%. What do they mean?

In [ ]:
name = "ehr_sahlgrenska_lab"
if have(name):
    sahl_vitals = stats[stats["source"] == name]
    print(f"vital-sign candidates in {name}: {len(sahl_vitals)}")
    # Where could the unit be? Look at the most common lab tests with their term_id, method and reference range.
    top_terms = count_by(name, ["term_name"]).head(15)["term_name"].tolist()
    sample = count_by(name, ["term_name", "term_id", "method", "ref_val1"], ds.field("term_name").isin(top_terms))
    display(show(sample.groupby("term_name", group_keys=False).head(3)))
    # Same test at other hospitals: does the Sahlgrenska value range match theirs?
    cmp_terms = ["B-Hemoglobin", "P/S-Kreatinin", "P-Kreatinin", "B-Leukocyter", "P-CRP", "P-Natrium", "P-Kalium"]
    rows = []
    for src in ["ehr_sahlgrenska_lab", "ehr_sa_vgr_lab", "ehr_karolinska_lab", "ehr_danderyd_lab_meas"]:
        if not path(src).exists():
            continue
        cols = pq.read_schema(path(src)).names
        df = pd.concat(scan(src, ["term_name", "value_num"] + (["unit"] if "unit" in cols else []),
                            ds.field("term_name").isin(cmp_terms)), ignore_index=True)
        if "unit" not in df:
            df["unit"] = pd.NA
        g = df.groupby(["term_name", "unit"], dropna=False)["value_num"]
        r = g.quantile([0.05, 0.5, 0.95]).unstack().add_prefix("q")
        r["n"] = g.size()
        r = r.reset_index()
        r.insert(0, "source", src)
        rows.append(r)
    display(show(pd.concat(rows).sort_values(["term_name", "source"])))

## 5. Death register: variable list against our file

I compared Socialstyrelsen's variable list (`register-dodsorsaksregistret-variabelforteckning.xls`) with the schema of `sos_deathreg_wide` and the profile.

**What our file covers**

| Our column | Register variable | What to know |
|---|---|---|
| `lopnr_swed` | PNR (pseudonymised) | Should be one row per person. Checked below. |
| `date_death_minimp`, `date_death_meanimp`, `inexact_death_date` | DODSDATN / DODSDAT | SoS: ~1–2% incomplete dates since 1979, more from 2015. The two columns look like two imputations (earliest possible vs midpoint). That's my reading; confirm it. |
| `ULORSAK` | underlying cause of death (ICD) | SoS says 0.5–1.5% missing from 1992, but **our file has 10% null**. Checked by year below. |
| `MORSAK1–48` | contributing causes | Number used varies by year. |
| `KAP19` | main injury for external causes (1997–) | 95% null is expected: only filled for injuries and poisonings. |
| `DODSPL` | place of death | 2003– coding: 1 = hospital, 2 = nursing home/special housing, 3 = private home, 4 = other/unknown. SoS: large gaps 2003–2006, ~10% from 2007. Our file: 18% null. |
| `DBGRUND1` | autopsy | 1 = clinical, 2 = extended forensic (ended 2012), 3 = forensic, 4 = forensic external examination, blank = none. |
| `LK` | municipality of residence | |
| `is_female` | KON | |

Deaths run **2003-01-01 to 2017-12-31**, so every cause is **ICD-10** (Sweden switched in 1997). The old DODSPL codes from 1987–1990 don't apply.

**Not in our file:** ALDER (age at death; compute it from the birth date), the flags ALKOHOL / NARKOTIK / DIABETES, OPERERAD/OPDAT, DODUTL/FOBUTL (died abroad / not registered in Sweden), RO1–48 (where on the certificate each diagnosis was written), DBGRUND5–8, the substance register SUBSTANS1–48 (2012–), and DOD_KOMMUN.

**Coverage:** since 2012 the register also includes people who died in Sweden without being registered here, added about 15 months late. Check whether that shows up as fewer deaths or more missing causes in the last years.

In [ ]:
name = "sos_deathreg_wide"
if have(name):
    info(name)
    dr = ds.dataset(path(name), format="parquet").to_table(filter=not_excluded()).to_pandas()
    dup = dr["lopnr_swed"].duplicated(keep=False)
    print(f"\nrows {len(dr):,} | persons {dr['lopnr_swed'].nunique():,} | rows of persons with >1 row {dup.sum():,}")
    print("from:", dr["from"].value_counts(dropna=False).to_dict())

In [ ]:
if have(name):
    mors = [c for c in dr.columns if c.startswith("MORSAK")]
    dr["year"] = pd.to_datetime(dr["date_death_minimp"]).dt.year
    dr["n_morsak"] = dr[mors].notna().sum(axis=1)
    dr["icd10_like"] = dr["ULORSAK"].str.match(r"^[A-Z]\d{2}", na=False)
    by_year = dr.groupby("year").agg(
        n=("lopnr_swed", "size"),
        ulorsak_null_pct=("ULORSAK", lambda s: 100 * s.isna().mean()),
        ulorsak_not_icd10_pct=("icd10_like", lambda s: 100 * (1 - s.mean())),
        dodspl_null_pct=("DODSPL", lambda s: 100 * s.isna().mean()),
        inexact_date_pct=("inexact_death_date", lambda s: 100 * s.mean()),
        morsak_mean=("n_morsak", "mean"),
        morsak_max=("n_morsak", "max"),
        autopsy_pct=("DBGRUND1", lambda s: 100 * s.notna().mean()),
    ).round(2).reset_index()
    display(show(by_year, keys=["year"]))
    display(show(dr["DODSPL"].value_counts(dropna=False).rename("n").reset_index(), keys=["DODSPL"]))
    display(show(dr["DBGRUND1"].value_counts(dropna=False).rename("n").reset_index(), keys=["DBGRUND1"]))

In [ ]:
# Are the ULORSAK-null deaths a special group? Compare with SCB's death file if we have it.
if have(name) and have("scb_death"):
    scb = pq.read_table(path("scb_death"), columns=["lopnr_swed", "date_death_scb", "sos_scb_mismatch"]).to_pandas()
    m = dr[["lopnr_swed", "date_death_minimp", "ULORSAK"]].merge(scb, on="lopnr_swed", how="outer", indicator=True)
    m["ulorsak_null"] = m["ULORSAK"].isna() & (m["_merge"] != "right_only")
    display(show(m.groupby(["_merge", "ulorsak_null"]).size().rename("n").reset_index()))

## 6. Diagnoses from the patient register

`sos_patreg_diag_long` has **146 million rows** (one row per diagnosis) and `sos_patreg_wide` 62 million (one row per visit). The next cell estimates how much memory a load needs, so you know whether you need a fat node.

**Fat node.** On UPPMAX Bianca you get a large node through the interactive queue. Check the exact flags on your cluster: `-C mem256GB` and `-C mem512GB` are Bianca's names, and other clusters differ.

```bash
interactive -A sens2020598 -p node -C mem256GB -t 8:00:00
module load python      # or the module you normally use; `module spider python` lists versions
jupyter lab             # inside the ThinLinc desktop
```

Ways to reduce memory, in order of effect:

1. Read only the columns you need (`columns=`).
2. Read diagnosis codes as a **dictionary**, i.e. pandas `category` (`read_dictionary=`). There are 15 000 distinct codes over 146 M rows.
3. Filter at read time: years (`INDATUM`), care level, or your cohort's `lopnr_swed`.
4. `sos_patreg_wide_lean` is already dictionary-encoded and smaller.

In [ ]:
def load_estimate_gb(name, columns):
    # Uncompressed Parquet size of the chosen columns. Pandas with object strings needs roughly 2-4x that.
    m = pq.ParquetFile(path(name)).metadata
    idx = [i for i in range(m.num_columns) if m.schema.column(i).name in columns]
    return sum(m.row_group(r).column(i).total_uncompressed_size for r in range(m.num_row_groups) for i in idx) / 1e9

def ram_gb():
    slurm = os.environ.get("SLURM_MEM_PER_NODE")
    if slurm:
        return int(slurm) / 1e3
    return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9

DIAG = "sos_patreg_diag_long"
DIAG_COLS = ["lopnr_swed", "INDATUM", "care_level", "diag_pos", "icd10_code"]
if have(DIAG):
    info(DIAG)
    est = load_estimate_gb(DIAG, DIAG_COLS)
    print(f"\nuncompressed {est:.1f} GB -> expect ~{2 * est:.0f}-{4 * est:.0f} GB in pandas | node RAM {ram_gb():.0f} GB")

In [ ]:
# Optional: only diagnoses for persons in the ED cohort, and from a start date.
COHORT_ONLY = True
FROM_DATE = date(2010, 1, 1)

filt = ds.field("INDATUM") >= pa.scalar(FROM_DATE, pa.date32())
if COHORT_ONLY and path("swed_ed_core").exists():
    cohort = pc.unique(pq.read_table(path("swed_ed_core"), columns=["lopnr_swed"])["lopnr_swed"])
    filt = filt & ds.field("lopnr_swed").isin(cohort)
    print(f"cohort: {len(cohort):,} persons")
if EXCLUDE:
    filt = filt & not_excluded()

if have(DIAG):
    tbl = ds.dataset(path(DIAG), format="parquet").to_table(columns=DIAG_COLS, filter=filt)
    diag = tbl.to_pandas(strings_to_categorical=True)
    del tbl
    print(f"{len(diag):,} diagnosis rows | {diag.memory_usage(deep=True).sum() / 1e9:.1f} GB in pandas | peak {mem_gb():.1f} GB")
    diag.info()

In [ ]:
# Sanity checks. The profile lists codes like 'C15-C26' among the most common: are those real values or the profiler's grouping?
if have(DIAG):
    codes = diag["icd10_code"].astype("string")
    is_range = codes.str.contains("-", na=False).astype(bool)
    is_icd10 = codes.str.match(r"^[A-Z]\d{2}[0-9A-Z]{0,3}$", na=False).astype(bool)
    kind = np.select([codes.isna().to_numpy(), is_range.to_numpy(), is_icd10.to_numpy()], ["null", "range", "icd10"], "other")
    display(show(pd.Series(kind, name="code_kind").value_counts().rename("n").reset_index()))
    display(show(diag.groupby(["care_level", "diag_pos"], observed=True).size().rename("n").reset_index()))
    yr = pd.to_datetime(diag["INDATUM"]).dt.year
    display(show(yr.rename("year").value_counts().sort_index().rename("n").reset_index(), keys=["year"]))
    diag.to_parquet(OUT / "patreg_diag_cohort.parquet")   # stays on the cluster

## 7. Validate the synthetic dataset

The generator (`make_synthetic_swed.py`) was written from the profile. Its own `validation_report.csv` passes 634/634 checks, but those checks only compare **one column at a time** against the profile. Real data has links *between* columns and tables (age ↔ priority, ED visit → admission → death) that the profile never showed. This section measures them on both datasets.

1. Generate on the cluster: `python make_synthetic_swed.py --out-dir synthetic`
2. Run the cells below. They compute the same summary for real and synthetic, side by side.
3. Fix the generator where the differences matter for your experiments. Then export **only** the synthetic files and the suppressed report, after the project's export check.

In [ ]:
SYN = WORK / "synthetic"
PAIRS = {  # real -> synthetic
    "swed_ed_core": "synth_ed_core",
    "sos_patreg_wide": "synth_patreg_wide",
    "sos_deathreg_wide": "synth_deathreg_wide",
    "ehr_danderyd_lab_meas": "synth_danderyd_lab_meas",
}
SAMPLE_MOD = 20   # real data: use every person with lopnr % 20 == 0 (5%) to keep memory low

def load(name, columns, synthetic):
    if synthetic:
        return pq.read_table(SYN / f"{PAIRS[name]}.parquet", columns=columns).to_pandas()
    parts = [df[df["lopnr_swed"] % SAMPLE_MOD == 0] for df in scan(name, columns, not_excluded())]
    return pd.concat(parts, ignore_index=True)

def marginals(df):
    rows = []
    for c in df.columns:
        s = df[c]
        r = {"column": c, "null_%": round(100 * s.isna().mean(), 2)}
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            q = s.quantile([0.05, 0.5, 0.95])
            r.update(q05=q.iloc[0], q50=q.iloc[1], q95=q.iloc[2])
        else:
            vc = s.value_counts(normalize=True).head(3)
            r["top3"] = ", ".join(f"{k}:{v:.1%}" for k, v in vc.items())
        rows.append(r)
    return pd.DataFrame(rows).set_index("column")

In [ ]:
ED_COLS = ["lopnr_swed", "ed_hospital", "ed_arrival_datetime", "ed_arrival_date", "ed_age", "is_male",
           "ed_priority_in", "ed_arrival_type", "ed_time_total_minutes", "ed_exit_to"]

def joint_checks(synthetic):
    ed = load("swed_ed_core", ED_COLS, synthetic)
    death = load("sos_deathreg_wide", ["lopnr_swed", "date_death_minimp"], synthetic)
    pr = load("sos_patreg_wide", ["lopnr_swed", "INDATUM", "care_level"], synthetic)
    out = {}
    per = ed.groupby("lopnr_swed").size()
    out["ED visits/person, mean"] = per.mean()
    out["persons with 1 ED visit, %"] = 100 * (per == 1).mean()
    for p, a in ed.groupby("ed_priority_in")["ed_age"].median().items():
        out[f"median age, priority {p:g}"] = a
    out["ambulance share by age>=80, %"] = 100 * ed.loc[ed["ed_age"] >= 80, "ed_arrival_type"].astype(str).str.contains("ambul", case=False).mean()
    # 30-day mortality after an ED visit
    m = ed[["lopnr_swed", "ed_arrival_date", "ed_priority_in"]].merge(death, on="lopnr_swed", how="left")
    days = (pd.to_datetime(m["date_death_minimp"]) - pd.to_datetime(m["ed_arrival_date"])).dt.days
    m["d30"] = days.between(0, 30)
    out["30-day mortality after ED visit, %"] = 100 * m["d30"].mean()
    for p, v in m.groupby("ed_priority_in")["d30"].mean().items():
        out[f"30-day mortality, priority {p:g}, %"] = 100 * v
    out["deaths before an ED visit (should be 0), %"] = 100 * (days < 0).mean()
    # ED visit followed by a patient-register episode the same or next day
    e = ed[["lopnr_swed", "ed_arrival_date"]].drop_duplicates()
    j = e.merge(pr[["lopnr_swed", "INDATUM", "care_level"]], on="lopnr_swed")
    j["lag"] = (pd.to_datetime(j["INDATUM"]) - pd.to_datetime(j["ed_arrival_date"])).dt.days
    hit = j[j["lag"].between(0, 1)].drop_duplicates(["lopnr_swed", "ed_arrival_date"])
    out["ED visits with patreg episode within 1 day, %"] = 100 * len(hit) / len(e)
    return pd.Series(out), marginals(ed)

real_joint, real_marg = joint_checks(synthetic=False)
syn_joint, syn_marg = joint_checks(synthetic=True)
report = pd.DataFrame({"real (5% sample)": real_joint, "synthetic": syn_joint})
report["diff"] = report["synthetic"] - report["real (5% sample)"]
report.round(2)

In [ ]:
# Vital signs: are values correlated within one measurement round in the real data? The generator draws them independently.
V = {"Pulsfrekvens": "pulse", "Andningsfrekvens": "resp", "Kroppstemperatur": "temp", "Saturation": "spo2",
     "Blodtryck systoliskt - övre": "sbp"}

def vital_corr(synthetic):
    cols = ["lopnr_swed", "datetime_analysis", "term_name", "value_num"]
    if synthetic:
        df = ds.dataset(SYN / f"{PAIRS['ehr_danderyd_lab_meas']}.parquet", format="parquet").to_table(
            columns=cols, filter=ds.field("term_name").isin(list(V))).to_pandas()
    else:
        filt = ds.field("term_name").isin(list(V))
        if EXCLUDE:
            filt = filt & not_excluded()
        df = pd.concat([d[d["lopnr_swed"] % SAMPLE_MOD == 0]
                        for d in scan("ehr_danderyd_lab_meas", cols, filt)], ignore_index=True)
    w = df.pivot_table(index=["lopnr_swed", "datetime_analysis"], columns="term_name", values="value_num", aggfunc="first")
    return w.rename(columns=V).corr(method="spearman").round(2)

print("real (5% sample)"); display(vital_corr(False))
print("synthetic");        display(vital_corr(True))

In [ ]:
# Column by column, side by side.
side = real_marg.join(syn_marg, lsuffix="_real", rsuffix="_syn")
side[sorted(side.columns)]

In [ ]:
# Privacy: no synthetic ED row should equal a real one on quasi-identifiers. A handful of chance hits is fine; many means copying.
KEYS = ["ed_hospital", "ed_arrival_datetime", "ed_age", "is_male"]
syn_ed = pq.read_table(SYN / "synth_ed_core.parquet", columns=KEYS).to_pandas().drop_duplicates()
hits = pd.concat([df.merge(syn_ed, on=KEYS) for df in scan("swed_ed_core", KEYS)]).drop_duplicates()
print(f"synthetic ED rows matching a real row exactly on {KEYS}: {len(hits):,} of {len(syn_ed):,}")

In [ ]:
# Save what may leave the cluster (after the export check): aggregates only, no lopnr.
report.round(2).to_csv(OUT / "synthetic_vs_real_joint.csv")
side.to_csv(OUT / "synthetic_vs_real_marginals.csv")
print(sorted(p.name for p in OUT.iterdir()))